# PMITC v1.0: load, join, plot a case
Labels are allegations by the named sources ("flagged by <source>"), not findings. The tables hold no personal names.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
DATA = '../data'
cases = pd.read_parquet(f'{DATA}/cases.parquet')
claims = pd.read_parquet(f'{DATA}/claims.parquet')
wallets = pd.read_parquet(f'{DATA}/wallets.parquet')
markets = pd.read_parquet(f'{DATA}/markets.parquet')
cases[['case_id','case_grade','investigation_outcome','n_claims','n_wallets_documented','benchmark_split']].sort_values('n_wallets_documented', ascending=False).head(15)

In [ ]:
# claims per grade and stance
pd.crosstab(claims.evidence_grade, claims.stance)

In [ ]:
# documented wallets (suspected and trade-confirmed) per case
wallets[wallets.is_documented].groupby('case_id').wallet.nunique().sort_values(ascending=False)

## One case: the Maduro market
Minute price path of the Yes token around both news times, with the pre-news buys of documented wallets.

In [ ]:
CASE = 'maduro-2026-01'
m = markets[markets.case_id == CASE]
mm = pd.read_parquet(f'{DATA}/market_minutes.parquet', filters=[('case_id','==',CASE), ('market_set','==','case')])
cid = mm[mm.t_news_version=='precise'].groupby('condition_id').usdc.sum().idxmax()
print(m.set_index('condition_id').loc[cid, ['question','t_news_utc','t_news_precise_utc','t_news_precise_precision']])

In [ ]:
pos = pd.read_parquet(f'{DATA}/positions.parquet', filters=[('case_id','==',CASE), ('condition_id','==',cid)])
print(pos.groupby('t_news_version').agg(buyers=('wallet','nunique'), documented=('is_documented','sum'), controls=('is_control','sum')))
fig, ax = plt.subplots(figsize=(10,4))
for v, g in mm[mm.condition_id == cid].groupby('t_news_version'):
    ax.plot(g.minute_utc, g.last_price, label=f'last price ({v} window)', alpha=.7)
row = m.set_index('condition_id').loc[cid]
ax.axvline(row.t_news_utc, color='k', ls='--', label='t_news (day)'); ax.axvline(row.t_news_precise_utc, color='r', ls='--', label='t_news_precise')
d = pos[(pos.t_news_version=='precise') & pos.is_documented]
ax.scatter(pd.to_datetime(d.first_buy_ts, unit='s', utc=True), d.first_buy_price, color='m', zorder=5, label='first buy, documented wallets')
ax.legend(fontsize=8); ax.set_ylabel('price'); ax.set_title(CASE); plt.tight_layout()

## Joins
`positions` (all pre-news buyers) joins `wallets` on (wallet, case_id), `markets` on (case_id, condition_id), and `fills` on (condition_id, asset_id, wallet) with `ts < t_cut_ts`.

In [ ]:
import duckdb
duckdb.sql(f"""SELECT market_set, source, count(*) legs FROM read_parquet('{DATA}/fills_part*.parquet') GROUP BY ALL ORDER BY 1, 2""").df()